# 🩺 Student Health Risk — LGBM + XGB + CatBoost Ensemble, Balanced-Accuracy Optimized

**Playground Series S6E7** — predict `health_condition` ∈ {`at-risk`, `unhealthy`, `fit`}, scored by **balanced accuracy**.

### Why this pipeline
The target is heavily imbalanced (`at-risk` ≈ 86%, `unhealthy` ≈ 8%, `fit` ≈ 6%), but balanced accuracy
weights every class **equally** — a plain accuracy-optimized model that over-predicts `at-risk` scores terribly.
Three things matter most here:

1. **Class-balanced training** — every model is trained with balanced class/sample weights so minority-class
   recall is not sacrificed.
2. **Informative missingness** — ~1–12% NaNs per column in this synthetic dataset carry signal. We keep NaNs
   native for the trees *and* add per-column missing indicators + a row-wise missing count.
3. **Decision-rule tuning** — balanced accuracy is maximized by tuning **per-class probability multipliers**
   on out-of-fold predictions (a smarter argmax), which is reliably worth several ×0.001 over raw argmax.

On top: a 3-model ensemble (LightGBM / XGBoost / CatBoost), 5-fold stratified CV, log-loss-optimal blend
weights, and a fine per-class multiplier grid — all tuned purely on OOF, so CV stays honest.


In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize

import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

class CFG:
    n_folds     = 5
    seed        = 42
    es_rounds   = 200
    data_dirs   = ['/kaggle/input/competitions/playground-series-s6e7',
                   '/kaggle/input/playground-series-s6e7',
                   'data']

DATA_DIR = next(d for d in CFG.data_dirs if os.path.exists(os.path.join(d, 'train.csv')))
print('Using data dir:', DATA_DIR)


In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

TARGET  = 'health_condition'
CLASSES = ['at-risk', 'fit', 'unhealthy']          # index = encoded label
CAT_COLS = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']
NUM_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']

y = train[TARGET].map({c: i for i, c in enumerate(CLASSES)}).values

print(train.shape, test.shape)
print(train[TARGET].value_counts(normalize=True).round(4))


## Feature engineering

Trees find univariate splits on their own, so we only add features they *can't* easily construct:
missing-value indicators (missingness is informative in this synthetic data), a row missing count,
and a few physiologically sensible ratios / interactions (calories-per-step, exercise intensity,
activity score, BMI × heart-rate load). NaNs stay NaN for LightGBM/XGBoost; CatBoost gets
categorical NaNs as an explicit `"missing"` level.


In [ ]:
def build_features(df):
    X = df[NUM_COLS + CAT_COLS].copy()

    # --- missingness signal ---
    X['n_missing'] = X[NUM_COLS + CAT_COLS].isna().sum(axis=1).astype(np.int8)
    for c in NUM_COLS + CAT_COLS:
        X[f'{c}_na'] = df[c].isna().astype(np.int8)

    # --- ratios & interactions ---
    X['cal_per_step']   = X['calorie_expenditure'] / (X['step_count'] + 1)
    X['cal_per_exmin']  = X['calorie_expenditure'] / (X['exercise_duration'] + 1)
    X['steps_x_ex']     = X['step_count'] * X['exercise_duration']
    X['hr_x_bmi']       = X['heart_rate'] * X['bmi']
    X['sleep_x_water']  = X['sleep_duration'] * X['water_intake']
    X['activity_score'] = X['step_count'] / 15000 + X['exercise_duration'] / 100
    X['bmi_cat']        = pd.cut(X['bmi'], [0, 18.5, 25, 30, 100], labels=False)

    for c in CAT_COLS:
        X[c] = X[c].astype('category')
    return X

X      = build_features(train)
X_test = build_features(test)
FEATURES = list(X.columns)
print(len(FEATURES), 'features')


## Cross-validated training — LightGBM, XGBoost, CatBoost

Each model runs the same 5-fold stratified CV with early stopping; we collect out-of-fold probabilities
(for honest tuning) and fold-averaged test probabilities. All three are trained class-balanced.


In [ ]:
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
FOLDS = list(skf.split(X, y))

def run_cv(name, fit_predict):
    """fit_predict(X_tr, y_tr, X_va, y_va, X_te) -> (va_proba, te_proba)"""
    t0 = time.time()
    oof  = np.zeros((len(X), 3))
    pt   = np.zeros((len(X_test), 3))
    for fold, (ti, vi) in enumerate(FOLDS):
        va_p, te_p = fit_predict(X.iloc[ti], y[ti], X.iloc[vi], y[vi], X_test)
        oof[vi] = va_p
        pt     += te_p / CFG.n_folds
        print(f'  fold {fold}: bal_acc={balanced_accuracy_score(y[vi], va_p.argmax(1)):.5f}')
    print(f'{name}: OOF bal_acc={balanced_accuracy_score(y, oof.argmax(1)):.5f} '
          f'logloss={log_loss(y, oof):.5f}  [{time.time()-t0:.0f}s]')
    return oof, pt


In [ ]:
# ---------------- LightGBM ----------------
def fit_lgb(X_tr, y_tr, X_va, y_va, X_te):
    m = lgb.LGBMClassifier(
        objective='multiclass', num_class=3,
        n_estimators=6000, learning_rate=0.03,
        num_leaves=127, min_child_samples=40,
        colsample_bytree=0.8, subsample=0.9, subsample_freq=1,
        reg_alpha=1.0, reg_lambda=5.0,
        class_weight='balanced',
        random_state=CFG.seed, n_jobs=-1, verbose=-1)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(CFG.es_rounds, verbose=False)])
    return m.predict_proba(X_va), m.predict_proba(X_te)

oof_lgb, pt_lgb = run_cv('LightGBM', fit_lgb)


In [ ]:
# ---------------- XGBoost ----------------
def fit_xgb(X_tr, y_tr, X_va, y_va, X_te):
    m = XGBClassifier(
        objective='multi:softprob', num_class=3,
        n_estimators=6000, learning_rate=0.03,
        max_depth=8, min_child_weight=20,
        colsample_bytree=0.8, subsample=0.9,
        reg_alpha=1.0, reg_lambda=5.0,
        tree_method='hist', enable_categorical=True,
        early_stopping_rounds=CFG.es_rounds, eval_metric='mlogloss',
        random_state=CFG.seed, n_jobs=-1, verbosity=0)
    sw = compute_sample_weight('balanced', y_tr)
    m.fit(X_tr, y_tr, sample_weight=sw, eval_set=[(X_va, y_va)], verbose=False)
    return m.predict_proba(X_va), m.predict_proba(X_te)

oof_xgb, pt_xgb = run_cv('XGBoost', fit_xgb)


In [ ]:
# ---------------- CatBoost ----------------
# CatBoost wants categorical NaNs as an explicit level
def cb_view(df):
    Xc = df.copy()
    for c in CAT_COLS:
        Xc[c] = Xc[c].cat.add_categories('missing').fillna('missing').astype(str)
    return Xc

Xc, Xc_test = cb_view(X), cb_view(X_test)

def fit_cat(X_tr, y_tr, X_va, y_va, X_te):
    m = CatBoostClassifier(
        loss_function='MultiClass', iterations=6000, learning_rate=0.06,
        depth=8, l2_leaf_reg=5.0, random_strength=1.0,
        auto_class_weights='Balanced',
        cat_features=CAT_COLS, od_type='Iter', od_wait=CFG.es_rounds,
        random_seed=CFG.seed, verbose=0, allow_writing_files=False)
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    return m.predict_proba(X_va), m.predict_proba(X_te)

def run_cv_cat():
    t0 = time.time()
    oof, pt = np.zeros((len(Xc), 3)), np.zeros((len(Xc_test), 3))
    for fold, (ti, vi) in enumerate(FOLDS):
        va_p, te_p = fit_cat(Xc.iloc[ti], y[ti], Xc.iloc[vi], y[vi], Xc_test)
        oof[vi] = va_p
        pt += te_p / CFG.n_folds
        print(f'  fold {fold}: bal_acc={balanced_accuracy_score(y[vi], va_p.argmax(1)):.5f}')
    print(f'CatBoost: OOF bal_acc={balanced_accuracy_score(y, oof.argmax(1)):.5f} '
          f'logloss={log_loss(y, oof):.5f}  [{time.time()-t0:.0f}s]')
    return oof, pt

oof_cat, pt_cat = run_cv_cat()


## Blend + decision-rule optimization

Three ensemble candidates, all built and evaluated purely on out-of-fold predictions:

1. **Equal-weight blend** of the three models' probabilities.
2. **Log-loss-optimal blend** — weights found by Nelder-Mead on OOF log loss (smooth surrogate).
3. **Logistic-regression stack** — a meta-model on the 9 OOF class probabilities, itself cross-validated
   with the same folds so its meta-OOF stays leak-free.

Then the key step for this metric — **per-class multipliers**: balanced accuracy is maximized not by
`argmax(p)` but by `argmax(w ⊙ p)`; we grid-search `w` (coarse → fine) on OOF for each candidate and
keep the best. This directly optimizes the competition metric while keeping CV honest.


In [ ]:
OOFS = [oof_lgb, oof_xgb, oof_cat]
PTS  = [pt_lgb,  pt_xgb,  pt_cat]

def blend(mats, w):
    w = np.asarray(w) / np.sum(w)
    return sum(wi * m for wi, m in zip(w, mats))

# --- log-loss optimal blend weights ---
def nll(w):
    w = np.abs(w)
    return log_loss(y, np.clip(blend(OOFS, w), 1e-15, 1))

res = minimize(nll, x0=np.ones(3) / 3, method='Nelder-Mead',
               options={'xatol': 1e-4, 'fatol': 1e-7, 'maxiter': 300})
w_ll = np.abs(res.x) / np.abs(res.x).sum()
print('log-loss blend weights (lgb, xgb, cat):', w_ll.round(4))

# --- per-class multiplier search: coarse log-spaced grid -> local refinement ---
# The optimal multipliers depend on how the models were weighted during training
# (an unweighted model would need w ~ 1/prior ~ (15, 10)), so the grid spans two
# orders of magnitude in log space and then refines around the best point.
def bal_acc_fast(y_true, pred, counts):
    hits = np.zeros(3)
    np.add.at(hits, y_true[pred == y_true], 1)
    return (hits / counts).mean()

CLASS_COUNTS = np.bincount(y, minlength=3)

def tune_class_weights(P, y_true, lo=0.15, hi=30.0, n=40, refine=3):
    counts = np.bincount(y_true, minlength=3)
    best = (1.0, 1.0)
    best_s = bal_acc_fast(y_true, P.argmax(1), counts)
    g1 = g2 = np.logspace(np.log10(lo), np.log10(hi), n)
    for _ in range(refine + 1):
        for w1 in g1:
            for w2 in g2:
                s = bal_acc_fast(y_true, (P * np.array([1.0, w1, w2])).argmax(1), counts)
                if s > best_s:
                    best_s, best = s, (w1, w2)
        c1, c2 = best
        r1 = (g1[1] / g1[0]) ** 2; r2 = (g2[1] / g2[0]) ** 2   # zoom around best
        g1 = np.logspace(np.log10(c1 / r1), np.log10(c1 * r1), 15)
        g2 = np.logspace(np.log10(c2 / r2), np.log10(c2 * r2), 15)
    return np.array([1.0, *best]), best_s


In [ ]:
# --- logistic-regression stack on OOF probabilities (leak-free via the same folds) ---
from sklearn.linear_model import LogisticRegression

M_oof  = np.hstack(OOFS)          # (n_train, 9)
M_test = np.hstack(PTS)           # (n_test, 9)

oof_stack = np.zeros((len(X), 3))
pt_stack  = np.zeros((len(X_test), 3))
for ti, vi in FOLDS:
    meta = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')
    meta.fit(M_oof[ti], y[ti])
    oof_stack[vi] = meta.predict_proba(M_oof[vi])
    pt_stack     += meta.predict_proba(M_test) / CFG.n_folds
print('stack OOF bal_acc (argmax):', round(balanced_accuracy_score(y, oof_stack.argmax(1)), 5))


In [ ]:
candidates = {
    'equal':   (blend(OOFS, np.ones(3) / 3), blend(PTS, np.ones(3) / 3)),
    'logloss': (blend(OOFS, w_ll),           blend(PTS, w_ll)),
    'stack':   (oof_stack,                   pt_stack),
}
results = {}
for name, (P_oof, P_te) in candidates.items():
    cw, s = tune_class_weights(P_oof, y)
    results[name] = (P_te, cw, s)
    print(f'{name:8s} -> raw={balanced_accuracy_score(y, P_oof.argmax(1)):.5f}  '
          f'tuned={s:.5f}  class_w={cw.round(3)}')

best_name = max(results, key=lambda k: results[k][2])
P_TEST, CLASS_W, CV_SCORE = results[best_name]
print(f'\nSelected: {best_name} | CV balanced accuracy = {CV_SCORE:.5f}')


In [ ]:
# --- individual model reference scores (tuned) ---
for name, o in zip(['LightGBM', 'XGBoost', 'CatBoost'], OOFS):
    cw, s = tune_class_weights(o, y, refine=1)
    print(f'{name:9s} tuned OOF bal_acc = {s:.5f}')


In [ ]:
pred = (P_TEST * CLASS_W).argmax(1)

sub[TARGET] = [CLASSES[i] for i in pred]
sub.to_csv('submission.csv', index=False)

print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()


### Notes
- CV here has tracked LB closely for most participants in this competition (see the *"Trust your CV"*
  discussion) — expect LB ≈ CV ± 0.001.
- Everything (blend weights **and** class multipliers) is tuned on out-of-fold predictions only,
  so the CV estimate is honest.
- Cheap extra juice if you have runtime to spare: bump `n_folds` to 10 and/or average 2–3 seeds.
